# 03 - Analise dos resultados

Este notebook responde as perguntas de negocio usando a camada silver.

In [ ]:
from pyspark.sql import functions as F

silver_table = "silver_acidentes"
silver_df = spark.table(silver_table).filter(F.col("data_acidente").isNotNull())
print(f"Linhas disponiveis para analise: {silver_df.count():,}")

## 1. Acidentes por dia da semana

In [ ]:
acidentes_por_dia = (
    silver_df.groupBy("dia_semana")
    .agg(
        F.count("*").alias("total_acidentes"),
        F.round(F.avg("gravidade"), 2).alias("gravidade_media"),
    )
    .orderBy(F.desc("total_acidentes"))
)
display(acidentes_por_dia)

## 2. Acidentes por horario

In [ ]:
acidentes_por_hora = (
    silver_df.filter(F.col("hora").isNotNull())
    .groupBy("hora")
    .agg(
        F.count("*").alias("total_acidentes"),
        F.round(F.avg("gravidade"), 2).alias("gravidade_media"),
    )
    .orderBy(F.desc("total_acidentes"))
)
display(acidentes_por_hora)

## 3. Acidentes por regiao

In [ ]:
acidentes_por_regiao = (
    silver_df.groupBy("regiao")
    .agg(
        F.count("*").alias("total_acidentes"),
        F.sum("feridos").alias("total_feridos"),
        F.sum("mortos").alias("total_mortos"),
        F.round(F.avg("gravidade"), 2).alias("gravidade_media"),
    )
    .orderBy(F.desc("total_acidentes"))
)
display(acidentes_por_regiao)

## 4. Evolucao anual

In [ ]:
acidentes_por_ano = (
    silver_df.filter(F.col("ano").isNotNull())
    .groupBy("ano")
    .agg(
        F.count("*").alias("total_acidentes"),
        F.sum("feridos").alias("total_feridos"),
        F.sum("mortos").alias("total_mortos"),
        F.round(F.avg("gravidade"), 2).alias("gravidade_media"),
    )
    .orderBy("ano")
)
display(acidentes_por_ano)

## 5. Ocorrencias mais graves

In [ ]:
acidentes_mais_graves = (
    silver_df.orderBy(F.desc("gravidade"), F.desc("mortos"), F.desc("feridos"))
    .select("data_acidente", "hora_acidente", "regiao", "feridos", "mortos", "gravidade")
    .limit(20)
)
display(acidentes_mais_graves)

## 6. Fatores contribuintes e gravidade

A analise considera os cinco fatores registrados para os veiculos envolvidos e remove valores sem especificacao.

In [ ]:
factor_columns = [
    "contributing_factor_vehicle_1",
    "contributing_factor_vehicle_2",
    "contributing_factor_vehicle_3",
    "contributing_factor_vehicle_4",
    "contributing_factor_vehicle_5",
]

fatores_df = silver_df.select(
    "gravidade",
    "feridos",
    "mortos",
    F.explode(F.array(*[F.col(column) for column in factor_columns])).alias("fator_contribuinte"),
)

fatores_df = fatores_df.filter(
    F.col("fator_contribuinte").isNotNull()
    & (F.trim(F.col("fator_contribuinte")) != "")
    & (F.lower(F.trim(F.col("fator_contribuinte"))) != "unspecified")
)

acidentes_por_fator = (
    fatores_df.groupBy("fator_contribuinte")
    .agg(
        F.count("*").alias("total_registros"),
        F.round(F.avg("gravidade"), 2).alias("gravidade_media"),
        F.sum("feridos").alias("total_feridos"),
        F.sum("mortos").alias("total_mortos"),
    )
    .orderBy(F.desc("gravidade_media"), F.desc("total_registros"))
)

display(acidentes_por_fator)

## 7. Persistencia dos resultados e qualidade

As tabelas gold abaixo serao usadas para documentar as respostas do MVP.

In [ ]:
gold_tables = {
    "gold_acidentes_por_dia": acidentes_por_dia,
    "gold_acidentes_por_hora": acidentes_por_hora,
    "gold_acidentes_por_regiao": acidentes_por_regiao,
    "gold_acidentes_por_ano": acidentes_por_ano,
    "gold_acidentes_por_fator": acidentes_por_fator,
}

for table_name, dataframe in gold_tables.items():
    dataframe.write.mode("overwrite").format("delta").saveAsTable(table_name)
    print(f"Tabela salva: {table_name}")

quality_checks = {
    "linhas_silver": silver_df.count(),
    "datas_nulas": silver_df.filter(F.col("data_acidente").isNull()).count(),
    "regioes_nulas": silver_df.filter(F.col("regiao").isNull()).count(),
    "gravidades_nulas": silver_df.filter(F.col("gravidade").isNull()).count(),
}

print("Controles de qualidade:")
for check_name, check_value in quality_checks.items():
    print(f"{check_name}: {check_value}")

if any(value != 0 for key, value in quality_checks.items() if key != "linhas_silver"):
    raise ValueError("A qualidade dos dados exige revisao antes da entrega.")

print("Todos os controles de qualidade passaram.")